# simpleNMR → JASON assignment write-back — version 2 (exact matching)

Writes the simpleNMR assignments in `input_py.json` back into the JASON `.jjh5` with
`Molecule.set_assignments()` (beautifuljason ≥ 1.3.0b1).

**Changes from version 1.** Every value in the JSON was exported from JASON, so nothing is approximated:

* **Atoms** – simpleNMR atom *n* must be JASON atom *n − 1*. The notebook checks that elements and bonds agree
  and **stops** if they don't (no graph-isomorphism guessing).
* **Spectra** – the 13C 1D and HSQC are found by matching the peak lists stored in the JSON
  (`oldjsondata`) exactly against each spectrum in the `.jjh5`.
* **Peaks** – each simpleNMR ppm value must equal a peak position in that spectrum (within `PPM_EPS`,
  a floating-point allowance, not a chemical tolerance). A value with no exact match is reported, not bridged.

The one remaining judgement call: for a CH₂ with two predicted shifts marked `(dn)`/`(up)` and two proton signals,
they are paired in ascending ppm order.

`DRY_RUN = True` by default. Review the report, set it to `False`, and re-run the last cell.

In [9]:
# ---------------------------------------------------------------- configuration
from pathlib import Path

SIMPLENMR_JSON = Path('input_py.json')   # simpleNMR output
JJH5_FILE      = None                    # None → input.jjh5 next to the JSON, then workingDirectory/workingFilename.jjh5
MOLECULE_INDEX = 0                       # zero-based molecule in the JASON document

DRY_RUN     = True     # True: report only.  False: write assignments into the .jjh5
MAKE_BACKUP = True     # copy the .jjh5 to <name>.<timestamp>.bak.jjh5 before writing

PPM_EPS = 1e-4         # floating-point allowance for "same value" (ppm)

In [10]:
# ---------------------------------------------------------------- imports
import json
import math
import shutil
from dataclasses import dataclass, field
from datetime import datetime

import pandas as pd
import beautifuljason as bjason

H1  = bjason.Molecule.Atom.NuclType.H1
C13 = bjason.Molecule.Atom.NuclType.C13
ELEMENT_H = int(bjason.Molecule.Atom.Type.H)

print('beautifuljason', getattr(bjason, '__version__', '?'),
      '| set_assignments available:', hasattr(bjason.Molecule, 'set_assignments'))

beautifuljason 1.3.0b1 | set_assignments available: True


## 1 · Read the simpleNMR JSON

In [11]:
@dataclass
class SnmrCarbon:
    idx: int                      # 0-based atom index (atomNumber − 1), same in JASON
    label: str                    # e.g. C7
    c_ppm: float | None           # experimental 13C
    h_ppms: list = field(default_factory=list)   # experimental 1H (1 or 2 values for a CH2)
    n_h: int = 0


def _value(v, limit):
    """float, or None for missing / simpleNMR's 10000 placeholder."""
    try:
        v = float(v)
    except (TypeError, ValueError):
        return None
    return v if math.isfinite(v) and abs(v) < limit else None


def load_simplenmr(path):
    data = json.loads(Path(path).read_text())
    atoms = data.get('catoms') or data.get('nodes_now')
    if not atoms:
        raise ValueError('No "catoms" / "nodes_now" in the simpleNMR JSON')

    carbons = []
    for a in atoms:
        if a.get('symbol') != 'C':
            continue
        h = [x for x in (_value(v, 100) for v in a.get('H1_ppm') or []) if x is not None]
        carbons.append(SnmrCarbon(idx=int(a['atomNumber']) - 1, label=f"C{a['atomNumber']}",
                                  c_ppm=_value(a.get('ppm'), 1000), h_ppms=sorted(h),
                                  n_h=int(a.get('numProtons') or 0)))
    carbons.sort(key=lambda c: c.idx)

    mg = data['molgraph']
    elements = {int(n['id']): int(n['atom_number']) for n in mg['nodes']}
    bonds = {frozenset((int(l['source']), int(l['target']))) for l in mg['links']}

    # experimental peak lists simpleNMR used, keyed by experiment (C13_1D, HSQC, ...)
    peak_lists = {}
    for key, block in (data.get('oldjsondata') or {}).items():
        if isinstance(block, dict) and block.get('datatype') == 'nmrspectrum':
            peaks = block.get('peaks', {}).get('data', {}).values()
            peak_lists[block.get('experimenttype', key)] = {
                'nuclei': block.get('nucleus'),
                'peaks': [(float(p['delta1']), float(p['delta2'])) for p in peaks],
            }
    return data, carbons, elements, bonds, peak_lists


def resolve_jjh5(json_path, data, explicit=None):
    candidates = [Path(explicit)] if explicit else []
    candidates.append(Path(json_path).parent / 'input.jjh5')
    if data.get('workingDirectory') and data.get('workingFilename'):
        candidates.append(Path(data['workingDirectory']) / f"{data['workingFilename']}.jjh5")
    for c in candidates:
        if c.expanduser().is_file():
            return c.expanduser().resolve()
    raise FileNotFoundError('No .jjh5 found; tried:\n  ' + '\n  '.join(map(str, candidates)))


snmr_data, snmr_carbons, snmr_elements, snmr_bonds, snmr_peak_lists = load_simplenmr(SIMPLENMR_JSON)
jjh5_path = resolve_jjh5(SIMPLENMR_JSON, snmr_data, JJH5_FILE)
print('SMILES :', snmr_data.get('smilesString'))
print('jjh5   :', jjh5_path)
print('spectra in JSON:', {k: len(v['peaks']) for k, v in snmr_peak_lists.items()})
pd.DataFrame([{'atom': c.label, 'nH': c.n_h, '13C': c.c_ppm,
               '1H': ', '.join(f'{h:.4f}' for h in c.h_ppms)} for c in snmr_carbons])

SMILES : CC(=O)C=CC1C(C)=CCCC1(C)C
jjh5   : /Users/vsmw51/Library/CloudStorage/OneDrive-DurhamUniversity/projects/programming/2025/python/awh/simpleNMRjeolTools2/archive/development/input.jjh5
spectra in JSON: {'C13_1D': 13, 'COSY': 9, 'HMBC': 16, 'HSQC': 11}


,atom,nH,13C,1H
0,C1,0,198.3108,
1,C2,1,148.9245,6.5926
2,C3,1,132.2711,6.0282
3,C4,0,131.8402,
4,C5,1,122.5945,5.4770
5,C6,1,54.2483,2.2607
6,C7,0,32.4580,
7,C8,2,31.1721,"1.2076, 1.4388"
8,C9,3,27.7398,0.8304
9,C10,3,26.8816,2.2309


## 2 · Checks and exact lookups

In [12]:
def check_atoms(molecule):
    """simpleNMR atom i must be JASON atom i: same element, same heavy-atom bonds."""
    jason_elements = {i: int(a.type) for i, a in enumerate(molecule.atoms) if int(a.type) != ELEMENT_H}
    jason_bonds = {frozenset((i, int(j)))
                   for i in jason_elements for j in molecule.atoms[i].bonded_atom_numbers
                   if int(j) in jason_elements}
    problems = []
    if jason_elements != snmr_elements:
        diff = sorted(set(jason_elements.items()) ^ set(snmr_elements.items()))
        problems.append(f'element/index differences (index, atomic number): {diff}')
    if jason_bonds != snmr_bonds:
        only_j = sorted(tuple(sorted(b)) for b in jason_bonds - snmr_bonds)
        only_s = sorted(tuple(sorted(b)) for b in snmr_bonds - jason_bonds)
        problems.append(f'bonds only in JASON {only_j}; only in simpleNMR {only_s}')
    if problems:
        raise ValueError('simpleNMR and JASON molecules do not match atom-for-atom:\n  ' + '\n  '.join(problems))


def _same(a, b):
    return abs(a - b) <= PPM_EPS


def _nuclides(spectrum):
    return tuple(spectrum.spec_info.nuclides)


def _peaks(spectrum):
    try:
        return list(spectrum.peaks)
    except KeyError:
        return []


def _json_peak_found(json_peak, jjh5_peaks, ndim):
    """JSON stores (delta1, delta2); compare in either dimension order."""
    if ndim == 1:
        return any(_same(float(p.pos[0]), json_peak[0]) for p in jjh5_peaks)
    a, b = json_peak
    return any((_same(float(p.pos[0]), a) and _same(float(p.pos[1]), b)) or
               (_same(float(p.pos[0]), b) and _same(float(p.pos[1]), a)) for p in jjh5_peaks)


def identify_spectrum(document, experiment, ndim, nuclei):
    """Index of the nmr_data spectrum whose peaks contain every peak simpleNMR stored for `experiment`."""
    if experiment not in snmr_peak_lists:
        return None, f'{experiment}: not in JSON'
    json_peaks = snmr_peak_lists[experiment]['peaks']
    results = []
    for i, s in enumerate(document.nmr_data):
        if s.ndim != ndim or set(_nuclides(s)) != set(nuclei):
            continue
        peaks = _peaks(s)
        found = sum(_json_peak_found(jp, peaks, ndim) for jp in json_peaks)
        results.append((found, i))
    if not results:
        return None, f'{experiment}: no {ndim}D {"/".join(nuclei)} spectrum in the jjh5'
    found, i = max(results)
    note = f'{experiment}: nmr_data[{i}] – {found}/{len(json_peaks)} JSON peaks found exactly'
    if found < len(json_peaks):
        note += '  ⚠ not all peaks match – has the spectrum been re-picked since the export?'
    return i, note


def exact_peaks(peaks, dim, value):
    return [p for p in peaks if _same(float(p.pos[dim]), value)]


def nearest_miss(peaks, dim, value):
    return min((abs(float(p.pos[dim]) - value) for p in peaks), default=None)


def predicted_shifts(molecule, nucleus):
    """{atom index: [(shift, predicted value, label)]} from the largest prediction spectrum; ignored shifts skipped.
    1H shifts are indexed by their carbon."""
    specs = [s for s in molecule.spectra if s.nucleus == nucleus]
    out = {}
    if not specs:
        return out
    spectrum = max(specs, key=lambda s: len(s.shifts))
    prefix = 'H' if nucleus == H1 else 'C'
    for shift in spectrum.shifts:
        if shift.ignored_auto or shift.ignored_user:
            continue
        value, _ = shift.get_value_error_pair(bjason.Molecule.CalcMethod.Best)
        nums = [int(n) for n in shift.nums]
        mark = f'({shift.mark})' if shift.mark else ''
        label = '/'.join(f'{prefix}{n + 1}{mark}' for n in nums)
        for n in nums:
            out.setdefault(n, []).append((shift, None if value is None else float(value), label))
    for lst in out.values():
        lst.sort(key=lambda t: (t[1] is None, t[1]))
    return out


def pair_shifts(shifts, signals):
    """Pair predicted shifts with signals, both in ascending ppm."""
    if not shifts or not signals:
        return []
    if len(shifts) == len(signals):
        return list(zip(shifts, signals))
    if len(shifts) == 1:
        return [(shifts[0], s) for s in signals]
    if len(signals) == 1:
        return [(sh, signals[0]) for sh in shifts]
    n = min(len(shifts), len(signals))
    return list(zip(shifts[:n], signals[:n]))

## 3 · Build the `(shift, signal[, dim])` tuples

In [13]:
def build_assignments(document, molecule_index=MOLECULE_INDEX):
    molecule = document.mol_data[molecule_index]
    check_atoms(molecule)

    c13_i, c13_note = identify_spectrum(document, 'C13_1D', 1, ('13C',))
    hsqc_i, hsqc_note = identify_spectrum(document, 'HSQC', 2, ('1H', '13C'))
    c13_peaks = _peaks(document.nmr_data[c13_i]) if c13_i is not None else []
    if hsqc_i is not None:
        hsqc_peaks = _peaks(document.nmr_data[hsqc_i])
        hd, cd = _nuclides(document.nmr_data[hsqc_i]).index('1H'), _nuclides(document.nmr_data[hsqc_i]).index('13C')
    else:
        hsqc_peaks, hd, cd = [], None, None

    pred_c, pred_h = predicted_shifts(molecule, C13), predicted_shifts(molecule, H1)
    tuples, rows, seen = [], [], set()

    def row(c, nucleus, label, predicted, snmr, jjh5, source, status):
        rows.append({'atom': c.label, 'nucleus': nucleus, 'shift': label, 'predicted': predicted,
                     'simpleNMR': snmr, 'jjh5 peak': jjh5, 'source': source, 'status': status})

    def add(tup):
        key = tuple(o if isinstance(o, int) else getattr(getattr(o, 'h5_group', None), 'name', id(o)) for o in tup)
        if key not in seen:
            seen.add(key)
            tuples.append(tup)

    for c in snmr_carbons:
        # ---------------- 13C: exact peak in the 13C 1D, else the HSQC carbon axis (if no 1D was used)
        c_shifts = pred_c.get(c.idx, [])
        if c.c_ppm is None:
            row(c, '13C', '', None, None, None, '', 'no 13C value in simpleNMR')
        else:
            if c13_peaks:
                hits, source, dim = exact_peaks(c13_peaks, 0, c.c_ppm), f'13C 1D [nmr_data {c13_i}]', None
            else:
                hits, source, dim = exact_peaks(hsqc_peaks, cd, c.c_ppm) if hsqc_peaks else [], f'HSQC [nmr_data {hsqc_i}]', cd
            if not c_shifts:
                row(c, '13C', '', None, c.c_ppm, None, source, 'no (non-ignored) predicted 13C shift')
            elif not hits:
                miss = nearest_miss(c13_peaks or hsqc_peaks, dim or 0, c.c_ppm)
                row(c, '13C', c_shifts[0][2], c_shifts[0][1], c.c_ppm, None, source,
                    f'NO EXACT PEAK (nearest {miss:.4f} ppm away)' if miss is not None else 'NO PEAKS')
            else:
                peak = hits[0]
                status = 'assigned' if len(hits) == 1 else f'assigned ({len(hits)} identical peaks, first used)'
                for sh, val, lab in c_shifts:
                    add((sh, peak) if dim is None else (sh, peak, dim))
                    row(c, '13C', lab, val, c.c_ppm, float(peak.pos[dim or 0]), source, status)

        # ---------------- 1H: exact HSQC 1H coordinate; carbon coordinate breaks ties
        if not c.h_ppms:
            continue
        signals = []
        for h in c.h_ppms:
            hits = exact_peaks(hsqc_peaks, hd, h) if hsqc_peaks else []
            if not hits:
                miss = nearest_miss(hsqc_peaks, hd, h) if hsqc_peaks else None
                row(c, '1H', '', None, h, None, f'HSQC [nmr_data {hsqc_i}]',
                    f'NO EXACT PEAK (nearest {miss:.4f} ppm away)' if miss is not None else 'NO HSQC PEAKS')
                continue
            if len(hits) > 1 and c.c_ppm is not None:
                hits.sort(key=lambda p: abs(float(p.pos[cd]) - c.c_ppm))
            signals.append((hits[0], h, len(hits)))
        h_shifts = pred_h.get(c.idx, [])
        if signals and not h_shifts:
            for peak, h, _ in signals:
                row(c, '1H', '', None, h, h, f'HSQC [nmr_data {hsqc_i}]', 'no (non-ignored) predicted 1H shift')
            continue
        note = f' ({len(h_shifts)} shift(s) ↔ {len(signals)} signal(s))' if signals and len(h_shifts) != len(signals) else ''
        for (sh, val, lab), (peak, h, n_hits) in pair_shifts(h_shifts, signals):
            add((sh, peak, hd))
            tie = f', {n_hits} peaks share this 1H – nearest in 13C used' if n_hits > 1 else ''
            row(c, '1H', lab, val, h, float(peak.pos[hd]), f'HSQC [nmr_data {hsqc_i}]', 'assigned' + note + tie)

    report = pd.DataFrame(rows)
    info = {'atoms': 'match atom-for-atom', 'spectra': [c13_note, hsqc_note], 'tuples': len(tuples)}
    return molecule, tuples, report, info

## 4 · Dry run (read-only)

In [14]:
with bjason.Document(str(jjh5_path), mode='r') as doc:
    _, _, report, info = build_assignments(doc)

print('atoms  :', info['atoms'])
for note in info['spectra']:
    print('spectra:', note)
print('tuples :', info['tuples'])
pd.set_option('display.float_format', '{:.4f}'.format)
report

atoms  : match atom-for-atom
spectra: C13_1D: nmr_data[4] – 13/13 JSON peaks found exactly
spectra: HSQC: nmr_data[1] – 11/11 JSON peaks found exactly
tuples : 25


,atom,nucleus,shift,predicted,simpleNMR,jjh5 peak,source,status
0,C1,13C,C1,197.2000,198.3108,198.3108,13C 1D [nmr_data 4],assigned
1,C2,13C,C2,148.2500,148.9245,148.9245,13C 1D [nmr_data 4],assigned
2,C2,1H,H2,6.5303,6.5926,6.5926,HSQC [nmr_data 1],assigned
3,C3,13C,C3,132.4500,132.2711,132.2711,13C 1D [nmr_data 4],assigned
4,C3,1H,H3,6.0900,6.0282,6.0282,HSQC [nmr_data 1],assigned
5,C4,13C,C4,132.0500,131.8402,131.8402,13C 1D [nmr_data 4],assigned
6,C5,13C,C5,122.5500,122.5945,122.5945,13C 1D [nmr_data 4],assigned
7,C5,1H,H5,5.4600,5.4770,5.4770,HSQC [nmr_data 1],assigned
8,C6,13C,C6,54.2500,54.2483,54.2483,13C 1D [nmr_data 4],assigned
9,C6,1H,H6,2.4888,2.2607,2.2607,HSQC [nmr_data 1],assigned


In [15]:
problems = report[~report['status'].str.startswith('assigned')] if not report.empty else report
print(f"{report['status'].str.startswith('assigned').sum()} rows assigned, {len(problems)} need attention")
problems

25 rows assigned, 0 need attention


,atom,nucleus,shift,predicted,simpleNMR,jjh5 peak,source,status


## 5 · Write the assignments
Set `DRY_RUN = False` in the configuration cell, re-run it, then run this cell.

In [16]:
DRY_RUN = False

In [17]:
if DRY_RUN:
    print('DRY_RUN is True – nothing written.')
else:
    if not hasattr(bjason.Molecule, 'set_assignments'):
        raise RuntimeError('This beautifuljason has no Molecule.set_assignments (needs ≥ 1.3.0b1)')
    if MAKE_BACKUP:
        backup = jjh5_path.with_name(f"{jjh5_path.stem}.{datetime.now():%Y%m%d-%H%M%S}.bak{jjh5_path.suffix}")
        shutil.copy2(jjh5_path, backup)
        print('Backup :', backup)
    # peak/shift objects belong to the open file, so rebuild inside the r+ session
    with bjason.Document(str(jjh5_path), mode='r+') as doc:
        molecule, tuples, report, info = build_assignments(doc)
        molecule.set_assignments(tuples)
    print(f"Wrote {info['tuples']} assignment(s) to {jjh5_path}")

Backup : /Users/vsmw51/Library/CloudStorage/OneDrive-DurhamUniversity/projects/programming/2025/python/awh/simpleNMRjeolTools2/archive/development/input.20260929-204255.bak.jjh5
Wrote 25 assignment(s) to /Users/vsmw51/Library/CloudStorage/OneDrive-DurhamUniversity/projects/programming/2025/python/awh/simpleNMRjeolTools2/archive/development/input.jjh5
